# HTEM 0 — Download experimental materials records

This non-workshop notebook is the online-data provenance step for the HTEM conductivity example. It uses the NREL HTEM API endpoints used in the course material. The public API can be slow or temporarily unavailable, so this is a resumable acquisition workflow; the workshop uses the frozen prepared Parquet release.


In [ ]:
from pathlib import Path
import time
import pandas as pd
import requests
from tqdm.auto import tqdm

OUTPUT_DIR = Path('outputs/htem_acquisition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LIBRARY_URL = 'https://htem-api.nrel.gov/api/sample_library'
SAMPLE_URL = 'https://htem-api.nrel.gov/api/sample/{}'
RUN_LIVE_COLLECTION = False
MAX_LIBRARIES = 5
REQUEST_DELAY_SECONDS = 1.0

def get_json(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY_SECONDS)
    return response.json()

if RUN_LIVE_COLLECTION:
    libraries = get_json(LIBRARY_URL)
    pd.DataFrame(libraries).to_parquet(OUTPUT_DIR / 'sample_libraries.parquet', index=False)
    print(f'Downloaded {len(libraries):,} library metadata records.')
else:
    print('Live collection disabled. Enable only for a planned API acquisition.')


In [ ]:
if RUN_LIVE_COLLECTION:
    rows, failures = [], []
    for library in tqdm(libraries[:MAX_LIBRARIES], desc='Downloading sample records'):
        library_detail = get_json(f"{LIBRARY_URL}/{library['id']}")
        for sample_id in library_detail.get('sample_ids', []):
            try:
                rows.append(get_json(SAMPLE_URL.format(sample_id)))
            except Exception as error:
                failures.append({'sample_id': sample_id, 'error': repr(error)})
    pd.DataFrame(rows).to_parquet(OUTPUT_DIR / 'htem_raw_samples.parquet', index=False)
    pd.DataFrame(failures).to_csv(OUTPUT_DIR / 'htem_download_failures.csv', index=False)
    print(f'Wrote {len(rows):,} raw sample records. Further optical interpolation and target carpentry are separate steps.')
